# Day 023 Project Solution — Article Scraper

A `BookScraper` that fetches books.toscrape.com, extracts structured data with CSS selectors, and provides AI insights about the catalog.

In [ ]:
import requests
import json
import ollama
from bs4 import BeautifulSoup


def parse_html(html_string: str) -> BeautifulSoup:
    return BeautifulSoup(html_string, "html.parser")


def find_all_links(soup: BeautifulSoup) -> list[dict]:
    links = []
    for a in soup.find_all("a", href=True):
        links.append({"text": a.get_text(strip=True), "href": a["href"]})
    return links


def extract_by_selector(soup: BeautifulSoup, css_selector: str) -> list[str]:
    return [
        el.get_text(strip=True)
        for el in soup.select(css_selector)
        if el.get_text(strip=True)
    ]


def fetch_and_parse(url: str, headers: dict | None = None) -> BeautifulSoup:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def ai_extract_from_page(html_content: str, question: str, model: str = "llama3.2") -> str:
    soup = BeautifulSoup(html_content, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a web page analyst. Answer questions about the page content concisely.",
            },
            {
                "role": "user",
                "content": f"Page content:\n{text[:3000]}\n\nQuestion: {question}",
            },
        ],
    )
    return response["message"]["content"]


class BookScraper:
    BASE = "https://books.toscrape.com"

    def __init__(self):
        self.session = requests.Session()
        self.session.headers["User-Agent"] = "Mozilla/5.0 (educational scraper)"

    def get_soup(self, url: str | None = None) -> BeautifulSoup:
        url = url or self.BASE
        r = self.session.get(url, timeout=10)
        r.raise_for_status()
        return BeautifulSoup(r.text, "html.parser")

    def extract_books(self, soup: BeautifulSoup) -> list[dict]:
        books = []
        for article in soup.select("article.product_pod"):
            title_tag = article.select_one("h3 a")
            price_tag = article.select_one("p.price_color")
            rating_tag = article.select_one("p.star-rating")
            books.append({
                "title": title_tag.get("title", "") if title_tag else "",
                "price": price_tag.get_text(strip=True) if price_tag else "",
                "rating": (rating_tag.get("class") or ["", ""])[1] if rating_tag else "",
            })
        return books

    def ai_insights(self, books: list[dict], question: str, model: str = "llama3.2") -> str:
        catalog = json.dumps(books[:10], indent=2)
        r = ollama.chat(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": "You are a book catalog analyst. Answer questions about the catalog concisely.",
                },
                {
                    "role": "user",
                    "content": f"Books:\n{catalog}\n\nQuestion: {question}",
                },
            ],
        )
        return r["message"]["content"]

## Action 1 — Fetch Front Page and Extract Books

In [ ]:
scraper = BookScraper()
soup = scraper.get_soup()
books = scraper.extract_books(soup)
print(f"Found {len(books)} books on the front page:")
for b in books[:5]:
    print(f"  {b['title'][:55]} | {b['price']} | {b['rating']}")

## Action 2 — Extract All Links from the Page

In [ ]:
links = find_all_links(soup)
print(f'\nFound {len(links)} links on the page')
# Show a few book links
book_links = [lk for lk in links if 'catalogue' in lk['href']][:3]
for lk in book_links:
    print(f"  {lk['href'][:60]}")

## Action 3 — AI Insights on the Catalog

In [ ]:
answer = scraper.ai_insights(
    books,
    'What price ranges do you see, and which rating appears most often?',
)
print('\nAI Insights:')
print(answer)
print('\nScraping complete!')